In [0]:
from pyspark.sql import functions as F
import logging
import sys

logger = logging.getLogger("turbines")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)
logger.propagate = False

In [0]:
storage_account = "sacuksnprdcdproject0001"

path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/project0001/nprd/turbine/landing/*.csv"

df = spark.read.option("header", True).csv(path)
df.show(5)
print(df.count())

logger.info(f"Read {df.count()} records from {path}")

In [0]:
df.printSchema()                  

In [0]:
df_casted = (
    df.withColumn("timestamp", F.col("timestamp").cast("timestamp"))
    .withColumn("turbine_id", F.col("turbine_id").cast("int"))
    .withColumn("wind_speed", F.col("wind_speed").cast("float"))
    .withColumn("wind_direction", F.col("wind_direction").cast("float"))
    .withColumn("power_output", F.col("power_output").cast("float"))
)
df_casted.show(5)
df_casted.printSchema()

In [0]:
null_counts = df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
).first().asDict()

total = df.count()
logger.info(f"Null check {null_counts}")

In [0]:
stats = (df.groupBy("turbine_id")
    .agg(
        F.min("power_output").alias("min_mw"),
        F.max("power_output").alias("max_mw"),
        F.round(F.avg("power_output"),2).alias("mean_mw"),
        F.round(F.stddev("power_output"),2).alias("std_mw"),
        F.count("*").alias("n"),
    )
    .withColumn("lower_threshold", F.round(F.col("mean_mw") - 2 * F.col("std_mw"),2))
    .withColumn("upper_threshold", F.round(F.col("mean_mw") + 2 * F.col("std_mw"),2))
)
stats.show()



In [0]:

df.groupBy("turbine_id", "timestamp").count().filter("count > 1").show()